# Notebook 01: Volatility Surface Construction via SVI
This notebook demonstrates how to construct and calibrate a continuous implied volatility surface from market option chain data using Gatheral's Stochastic Volatility Inspired (SVI) model.

### Steps:
1. Fetch live option chains from Yahoo Finance (`yfinance`).
2. Calculate Black-Scholes implied volatilities via Brent's root-finding method.
3. Filter Out-of-the-Money (OTM) options to remove duplicate strikes and noise.
4. Fit the SVI parametric model to each expiry slice.
5. Interpolate slices to build a continuous implied volatility surface $w(k, T)$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from dpp.data.market_data import fetch_option_chain_data
from dpp.calibration.implied_vol import implied_volatility_bs
from dpp.calibration.svi import fit_svi_slice, get_svi_surface

# 1. Fetch Option Data
ticker = "AAPL"
print(f"Fetching market option chains for {ticker}...")
df = fetch_option_chain_data(ticker)
print(f"Fetched {len(df)} contracts.")


In [ ]:
# 2. Calculate Implied Volatilities
df["implied_vol_calc"] = df.apply(
    lambda r: implied_volatility_bs(r["mid"], r["spot"], r["strike"], r["maturity"], 0.05, 0.0, r["option_type"]),
    axis=1
)
df = df[df["implied_vol_calc"] > 0.01].dropna().copy()
print(f"Contracts with valid IV: {len(df)}")


In [ ]:
# 3. Filter OTM-only options for calibration
df_otm = df[
    ((df["option_type"] == "call") & (df["strike"] >= df["spot"])) |
    ((df["option_type"] == "put") & (df["strike"] < df["spot"]))
].copy()
print(f"OTM contracts: {len(df_otm)}")


In [ ]:
# 4. Calibrate SVI for the longest expiry slice
expiries = df_otm["expiry_str"].unique()
target_expiry = expiries[-1]
slice_df = df_otm[df_otm["expiry_str"] == target_expiry].sort_values("strike")

strikes = slice_df["strike"].values
spot = slice_df["spot"].iloc[0]
k = np.log(strikes / spot)
T = slice_df["maturity"].iloc[0]
w_mkt = (slice_df["implied_vol_calc"].values ** 2) * T

# Fit SVI parameters
params = fit_svi_slice(k, w_mkt)
a, b, rho, m, sigma = params
w_pred = a + b * (rho * (k - m) + np.sqrt((k - m)**2 + sigma**2))

print(f"Fitted SVI parameters for T={T:.4f} ({target_expiry}):")
print(f"  a (variance level):  {a:.6f}")
print(f"  b (wing slope):      {b:.6f}")
print(f"  rho (skew/asymmetry): {rho:.6f}")
print(f"  m (location shift):  {m:.6f}")
print(f"  sigma (smoothing):   {sigma:.6f}")


In [ ]:
# 5. Plot SVI Smile Fit
plt.figure(figsize=(10, 6))
plt.scatter(k, np.sqrt(w_mkt/T), color='red', label='Market IV', alpha=0.7)
plt.plot(k, np.sqrt(w_pred/T), color='blue', label='SVI Fitted Smile', linewidth=2)
plt.title(f"SVI Fit for {ticker} (Expiry: {target_expiry}, T={T:.4f})")
plt.xlabel("Log-Moneyness k = log(K/S)")
plt.ylabel("Implied Volatility")
plt.legend()
plt.grid(True)
plt.show()
